In [1]:
import pandas as pd
import numpy as np

In [2]:
circuits_df = pd.read_csv('../input/datasets/rohanrao/formula-1-world-championship-1950-2020/circuits.csv')
pitstops_df = pd.read_csv('../input/datasets/rohanrao/formula-1-world-championship-1950-2020/pit_stops.csv')
qualifying_df = pd.read_csv('../input/datasets/rohanrao/formula-1-world-championship-1950-2020/qualifying.csv')
races_df = pd.read_csv('../input/datasets/rohanrao/formula-1-world-championship-1950-2020/races.csv')
results_df = pd.read_csv('../input/datasets/rohanrao/formula-1-world-championship-1950-2020/results.csv')
status_df = pd.read_csv('../input/datasets/rohanrao/formula-1-world-championship-1950-2020/status.csv')


In [3]:
# We Keep only those Circuits where races have happend in or after 2011
recent_races = races_df[races_df['year'] >=2011]
print(recent_races['circuitId'].value_counts())
# We keep races on those circuits which has more than 3 races 
recent_races = recent_races.groupby('circuitId').filter(lambda x: len(x) >= 3)
print(recent_races['circuitId'].nunique())

circuitId
9     15
4     14
11    14
14    14
24    14
3     14
13    14
70    13
6     13
18    13
1     12
7     12
69    12
22    12
15    12
17    10
32     9
71     8
73     8
2      7
10     5
21     4
34     4
77     4
39     4
5      3
68     3
79     3
35     3
20     3
78     3
12     2
75     2
80     2
76     1
Name: count, dtype: int64
31


In [4]:
# We merge step by step to avoid row multiplication
# Each race has multiple drivers (results), each driver has multiple pit stops, each qualifying session has multiple position entries.
recent_results = results_df[results_df['raceId'].isin(set(recent_races['raceId']))]
recent_results = recent_results.merge(recent_races[['raceId','circuitId','year']], on='raceId')
qualifying_df = qualifying_df.rename(columns={'position':'qualiPosition'})
recent_results = recent_results.merge(qualifying_df, on=['raceId','driverId'])
# Grid vs Qualifying position differece
recent_results['grid_drop'] = recent_results['grid'] - recent_results['qualiPosition']
# Only count where both are >0 (someone actually qualified and got a grid slot)
valid_qualis = recent_results[(recent_results['grid'] > 0) & (recent_results['qualiPosition'] > 0)]
print(recent_results.shape)
#No duplicate rows by (raceId, driverId)
print(recent_results.duplicated(subset=['raceId','driverId']).sum())

(5809, 28)
0


Feature Engineering

Status csv has 50 different values like finished, disquilaified,Accident,+1 Lap or Gearbox. But overall these can be broadly categorized into 4 categories: 1) Finish 2) Mechanical failures or DNFs 3) Accidental DNFs 4) Disqualified. By categorizing all the status can help us calculate meaningful features like: Mechinical dnf rate, accidental dnf rate that make a race more headline worthy and intreseting

In [5]:
def bin_status(status):
    if status == 'Finished' or status.startswith('+'):
        return 'finished'
    elif status in {'Disqualified','Excluded','107% Rule'}:
        return 'disqualified'
    elif status in {'Accident','Collision','Spun off','Fatal accident',
             'Collision damage','Debris','Damage'}:
        return 'accident'
    return 'mechanical'
status_df['statusBin'] = status_df['status'].apply(bin_status)
recent_results = recent_results.merge(status_df[['statusId','statusBin']], on='statusId', how='left')
# Shape after merge
print(recent_results.shape)

(5809, 29)


In [6]:
# We manually scrape some features as well
# length_km, n_corners
# Sources: F1.com, Wikipedia entries for each circuit (verify before final submission!)
manual = {
    9:  ('Silverstone',0, 5.891, 18),
    4:  ('Catalunya',0, 4.657, 14),
    3:  ('Bahrain',0, 5.412, 15),
    14: ('Monza',0, 5.793, 11),
    24: ('Yas Marina',0, 5.281, 16),
    11: ('Hungaroring',0, 4.381, 14),
    13: ('Spa',0, 7.004, 19),
    70: ('Red Bull Ring',0, 4.318, 10),
    18: ('Interlagos',0, 4.309, 15),
    6:  ('Monaco',1, 3.337, 19),
    1:  ('Albert Park',0, 5.278, 14),
    69: ('COTA',0, 5.513, 20),
    15: ('Marina Bay',1, 4.940, 19),
    22: ('Suzuka',0, 5.807, 18),
    7:  ('Gilles Villeneuve',0, 4.361, 14),
    17: ('Shanghai',0, 5.451, 16),
    32: ('Hermanos Rodriguez',0, 4.304, 17),
    71: ('Sochi',0, 5.848, 18),
    73: ('Baku',1, 6.003, 20),
    2:  ('Sepang',0, 5.543, 15),
    10: ('Hockenheim',0, 4.574, 17),
    39: ('Zandvoort',0, 4.259, 14),
    21: ('Imola',0, 4.909, 19),
    77: ('Jeddah',1, 6.174, 27),
    34: ('Paul Ricard',0, 5.842, 15),
    5:  ('Istanbul Park',0, 5.338, 14),
    20: ('Nurburgring (GP)',0, 5.148, 15),
    79: ('Miami',1, 5.412, 19),
    35: ('Korean (Yeongam)',0, 5.615, 18),
    68: ('Buddh',0, 5.125, 16),
    78: ('Losail',0, 5.419, 16),
}


In [7]:
unique_circuit_ids = recent_results['circuitId'].unique()

features = {}

for circuit_id in unique_circuit_ids:
    circuit_rows = recent_results[recent_results['circuitId'] == circuit_id]
    circuit_entries = len(circuit_rows)
    unique_races_by_circuit = circuit_rows['raceId'].unique()
    total_finishes = (circuit_rows['statusBin'] == 'finished').sum()
    total_disquali = (circuit_rows['statusBin'] == 'disqualified').sum()
    total_mech_dnf = (circuit_rows['statusBin'] == 'mechanical').sum()
    total_accident_dnf = (circuit_rows['statusBin'] == 'accident').sum()

    # Add DNF rate,mechnaical dnf rate, accidental Dnfs rate and disqualified rate
    dnf_rate = (circuit_entries - total_finishes)/circuit_entries
    mech_dnf_rate = total_mech_dnf/circuit_entries
    accident_dnf_rate = total_accident_dnf/circuit_entries
    disquali_rate = total_disquali/circuit_entries


    pole_to_wins = 0
    for race_id in unique_races_by_circuit:
        race_rows = circuit_rows[circuit_rows['raceId'] == race_id]
        winner_grid = race_rows[race_rows['positionOrder'] == 1]['grid'].values
        if len(winner_grid) and winner_grid[0] == 1:
                pole_to_wins += 1
    pole_win_rate = pole_to_wins / len(unique_races_by_circuit)

    # Start finish position changes
    # Circuit becomes interesting when there are overtakes and drivers pick up or loose positions
    circuit_finishes = circuit_rows[circuit_rows['statusBin'] == 'finished'].copy()
    # grid 0 means pit-lane start; treat as 21+ for position-change purposes
    circuit_finishes['grid'] = circuit_finishes['grid'].replace(0, circuit_finishes['grid'].max() + 1)
    circuit_finishes['positionDelta'] = circuit_finishes['grid'] - circuit_finishes['positionOrder']
    avg_pos_change = circuit_finishes['positionDelta'].mean()
    std_pos_change = circuit_finishes['positionDelta'].std() 

    valid_qualis_by_circuit = valid_qualis[valid_qualis['circuitId'] == circuit_id]
    if len(valid_qualis_by_circuit):
        avg_grid_drop = valid_qualis_by_circuit['grid_drop'].mean()
        big_penalty_rate = (valid_qualis_by_circuit['grid_drop'] >= 3).mean()
    else:
        avg_grid_drop = 0.0
        big_penalty_rate = 0.0
        
    label, street, length, corners = manual[circuit_id]
    
    features[circuit_id] = {
        'dnf_rate': dnf_rate,
        'mech_rate': mech_dnf_rate,
        'acc_rate': accident_dnf_rate,
        'dsq_rate': disquali_rate,
        'pole_win_rate': pole_win_rate,
        'avg_pos_change': avg_pos_change,
        'std_pos_change': std_pos_change,
        'avg_grid_drop':avg_grid_drop,
        'big_penalty_rate':big_penalty_rate,
        'label': label,
        'is_street' : street,
        'length_km' : length,
        'n_corners' : corners,
        'altitude': circuits_df.loc[circuits_df['circuitId'] == circuit_id, 'alt']

    }
    
    

In [8]:
df = pd.DataFrame(features).T
df = df.set_index('label')
print(df.round(3).to_string())
df.to_csv('circuit_features.csv')
print(f"\nSaved circuit_features.csv -- shape {df.shape}")

                    dnf_rate mech_rate  acc_rate  dsq_rate pole_win_rate avg_pos_change std_pos_change avg_grid_drop big_penalty_rate is_street length_km n_corners                            altitude
label                                                                                                                                                                                                  
Albert Park         0.290837  0.183267  0.095618  0.011952      0.333333       2.303371       3.688138     -0.069959         0.028807         0     5.278        14     0    10
Name: alt, dtype: int64
Sepang              0.222222  0.176471  0.045752       0.0      0.428571       2.042017       4.312762           0.0         0.026316         0     5.543        15     1    18
Name: alt, dtype: int64
Shanghai            0.113208  0.080189  0.033019       0.0           0.6       0.856383       3.626346           0.0         0.023585         0     5.451        16     16    5
Name: alt, dtype: int64
